# CNC

## Recearch: How does dxf work?
DXF files exist of a few sections, each section is divided into a header and a body. The header contains information about the section, while the body contains the actual data.
In each section you have elements of types:
- LINE
- CIRCLE
- ARC
- ELLIPSE
- POLYLINE
- TEXT
- POINT


Lets try and open a DXF file with Python.


In [ ]:
import ezdxf

file = "TestDXF.dxf"

def readAndPasteDXF(file):
    # Read the DXF file
    doc = ezdxf.readfile(file)
    
    # Iterate through the entities in the modelspace
    for entity in doc.modelspace().query('*'):
        if entity.dxftype() == 'LINE':
            print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
        elif entity.dxftype() == 'CIRCLE':
            print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
        elif entity.dxftype() == 'ARC':
            print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
        else:
            print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")


readAndPasteDXF(file)
        

Top! We kunnen de data uit een DXF bestand lezen met de `ezdxf` library. Wel zie ik dat (0,0,0) de oorsprong is van het bestand. Dit is niet handig, want we willen dat de oorsprong in het midden van de tekening zit. We kunnen dit oplossen door de tekening te verschuiven naar het midden.


In [ ]:
import ezdxf
import ezdxf.bbox

notCenteredFile = "NotCenterdDxf.dxf"

def getBoundBox(entity):
    # check if entity has a bounding box
    if hasattr(entity, 'bounding_box'):
        bounds = entity.bounding_box()
        return bounds
    # If the entity does not have a bounding box, return None
    print(f"Entity {entity.dxftype()} does not have a bounding box")

    # if circle, return a bounding box based on center and radius
    if entity.dxftype() == 'CIRCLE':
        center = entity.dxf.center
        radius = entity.dxf.radius
        return ezdxf.bbox.BoundingBox(
            min=(center.x - radius, center.y - radius),
            max=(center.x + radius, center.y + radius)
        )
    
    # if arc, return a bounding box based on center, radius and angles
    elif entity.dxftype() == 'ARC':
        center = entity.dxf.center
        radius = entity.dxf.radius
        start_angle = entity.dxf.start_angle
        end_angle = entity.dxf.end_angle
        
        # Calculate the bounding box based on the arc's extent
        min_x = center.x + radius * ezdxf.math.cos(start_angle)
        max_x = center.x + radius * ezdxf.math.cos(end_angle)
        min_y = center.y + radius * ezdxf.math.sin(start_angle)
        max_y = center.y + radius * ezdxf.math.sin(end_angle)
        
        return ezdxf.bbox.BoundingBox(
            
    else:
        return None


def center_dxf(filename):
    doc = ezdxf.readfile(filename)
    msp = doc.modelspace()
    
    # Find the bounding box of all entities
    min_x = min_y = float('inf')
    max_x = max_y = float('-inf')
    
    for entity in msp:
        if entity.dxftype() in {'LINE', 'CIRCLE', 'ARC', 'ELLIPSE', 'POLYLINE', 'TEXT', 'POINT'}:
            bbox = getBoundBox(entity)
            if bbox:
                min_x = min(min_x, bbox.extmin.x)
                min_y = min(min_y, bbox.extmin.y)
                max_x = max(max_x, bbox.extmax.x)
                max_y = max(max_y, bbox.extmax.y)

    # Calculate the center
    center_x = (min_x + max_x) / 2
    center_y = (min_y + max_y) / 2
    
    # Shift all entities to center them
    for entity in msp:
        if entity.dxftype() in {'LINE', 'CIRCLE', 'ARC', 'ELLIPSE', 'POLYLINE', 'TEXT', 'POINT'}:
            entity.translate(-center_x, -center_y)

    # Save the modified DXF file
    doc.saveas("centered_" + filename)
    
    
# print DXF file
doc = ezdxf.readfile(notCenteredFile)
for entity in doc.modelspace().query('*'):
    if entity.dxftype() == 'LINE':
        print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
    elif entity.dxftype() == 'CIRCLE':
        print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
    elif entity.dxftype() == 'ARC':
        print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
    else:
        print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")
        
# Center the DXF file
center_dxf(notCenteredFile)

# Print the centered DXF file
doc = ezdxf.readfile("centered_" + notCenteredFile)
for entity in doc.modelspace().query('*'):
    if entity.dxftype() == 'LINE':
        print(f"Line from {entity.dxf.start} to {entity.dxf.end}")
    elif entity.dxftype() == 'CIRCLE':
        print(f"Circle at {entity.dxf.center} with radius {entity.dxf.radius}")
    elif entity.dxftype() == 'ARC':
        print(f"Arc at {entity.dxf.center} with radius {entity.dxf.radius}, start angle {entity.dxf.start_angle}, end angle {entity.dxf.end_angle}")
    else:
        print(f"Entity type: {entity.dxftype()} with data: {entity.dxf}")

If we want the robot to draw the dxf, the robot should know a centerd position.
All variables should be definex based on that position

In [4]:
from gerrard import *
import time

zeroPosition = AbsPos(570,0,302.5,180,0,180) # 420
upvector = AbsPos(0,0,20,0,0,0)

newRobot = Robot("/dev/ttyUSB0")

newRobot.connect()

with newRobot:

    newRobot.end()
    newRobot.resetError()
    print(newRobot.executeCommand("SERVO ON", True))
    time.sleep(2)
    
    #newRobot.setAcceleration(50, 50)
    newRobot.overrideSpeed(50)

    newRobot.executeCommand("SPD M_NSPD", True)
    #newRobot.executeCommand("M_NSPD", True)
    for i in range(1, 7):
        newRobot.torqueLimit(i, 100)

    print("Code is gestart!")

    newRobot.setVariable("H", zeroPosition)
    newRobot.setVariable("HU", zeroPosition + upvector)
    # newRobot.setVariable("H1", AbsPos(270,0,400,180,0,180))

    newRobot.servoOn()

    time.sleep(1)

    newRobot.moveLinearTo("H", True, "P")

    time.sleep(2)

    newRobot.moveLinearTo("HU", True, "P")




Connecting
QoK3F;3F;7,0;3,5,A,1E,32,46,64;MB4;PRM;RV-3SB;CRn-5xx;MELFA;05-07-14;Ver.K4a;ENG;COPYRIGHT(C)1999-2005 MITSUBISHI ELECTRIC CORPORATION ALL RIGHTS RESERVED;1;1;8;
QeR601000000
QoKRLNG;1;1
QeR601000000
QoK1.MB4;9104;13-03-2222:04:26;2;215808;12;;4;155;135256;0;154;877
QeR601000000
QoKX;420.00;Y;0.00;Z;322.50;A;180.00;B;0.00;C;180.00;;7,0;100;0.00;00000000
QeR601000000
QoKMEXTL;0.00, 0.00, 0.00, 0.00, 0.00, 0.00;6
QeR601000000
QoK
Ended Connecting
QoK
Code is gestart!


Great! We can use a home position and move from there. The `upVector` can be used to move the cnc above the plate. Now, lets try moving it around

In [5]:
import ezdxf

file = "OnlyLinesDXF.dxf"
file = "House.dxf"

doc = ezdxf.readfile(file)

with newRobot:
    # Iterate through the LINE entities in the modelspace
    for entity in doc.modelspace().query('LINE'):
        print(f"Processing Line from {entity.dxf.start} to {entity.dxf.end}")
        start = entity.dxf.start
        end = entity.dxf.end
        
        newRobot.setVariable("L1", AbsPos(start.x*10, start.y*10, start.z*10, 0, 0, 0) + zeroPosition )
        time.sleep(0.2)
        newRobot.setVariable("LU1", AbsPos(start.x*10, start.y*10, start.z*10, 0, 0, 0) + zeroPosition + upvector)
        time.sleep(0.2)
        newRobot.setVariable("L2", AbsPos(end.x*10, end.y*10, end.z*10, 0, 0, 0) + zeroPosition)
        time.sleep(0.2)
        newRobot.setVariable("LU2", AbsPos(end.x*10, end.y*10, end.z*10, 0, 0, 0) + zeroPosition + upvector)
        time.sleep(0.2)

        newRobot.overrideSpeed(20)
        newRobot.moveLinearTo("LU1", True, "P")
        time.sleep(0.5)

        newRobot.overrideSpeed(10)
        time.sleep(0.5)
        newRobot.moveLinearTo("L1", True, "P")
        time.sleep(1)
        
        newRobot.moveLinearTo("L2", True, "P")
        time.sleep(3)
        newRobot.moveLinearTo("LU2", True, "P")
        time.sleep(0.5)


Processing Line from (-7.777157425880432, -3.882098197937012, 0.0) to (-7.777157425880432, -1.509734150022268, 0.0)
Processing Line from (-9.147856384515762, -3.882098197937012, 0.0) to (-9.147856384515762, -1.509734150022268, 0.0)
Processing Line from (-9.147856384515762, -1.509734150022268, 0.0) to (-7.777157425880432, -1.509734150022268, 0.0)
Processing Line from (-9.147856384515762, -3.882098197937012, 0.0) to (-7.777157425880432, -3.882098197937012, 0.0)
Processing Line from (-7.988034188747406, 1.099866535514593, 0.0) to (-7.988034188747406, 3.44587154686451, 0.0)
Processing Line from (-9.200575947761536, 1.099866535514593, 0.0) to (-9.200575947761536, 3.44587154686451, 0.0)
Processing Line from (-17.24557578563691, -4.5e-15, 0.0) to (-12.00000000000001, -5.000000000000002, 0.0)
Processing Line from (-12.0, 4.999999999999999, 0.0) to (-17.2455757856369, -3.5e-15, 0.0)
Processing Line from (-12.0, 5.000000000000001, 0.0) to (-12.00000000000001, -4.999999999999998, 0.0)
Processing 

Cool! A house. Lets refine the code a bit. Mayby make a frace class

In [1]:
from gerrard import Robot, AbsPos
import time
import ezdxf

class Frace(Robot):
    def __init__(self, port):
        super().__init__(port)
        self.zeroPosition = AbsPos(420, 0, 302.5, 180, 0, 180)
        self.upvector = AbsPos(0, 0, 20, 0, 0, 0)

    def setupRobot(self):
        with self:
            self.end()
            self.resetError()
            self.servoOn()
            time.sleep(2)

            self.overrideSpeed(50)

            self.executeCommand("SPD M_NSPD", True)
            for i in range(1, 7):
                self.torqueLimit(i, 50)

            time.sleep(1)

    def home(self):
        with self:
            self.setVariable("H", self.zeroPosition)
            self.setVariable("HU", self.zeroPosition + self.upvector)
            self.moveLinearTo("HU", True, "P")
            time.sleep(2)
            self.moveLinearTo("H", True, "P")
            time.sleep(2)
            self.moveLinearTo("HU", True, "P")
            time.sleep(2)
            self.servoOff()
            time.sleep(0.5)

    def drawLine(self, start, end):
        with self:
            print(f"Drawing line from {start} to {end} with length {self.calcLineLength(start, end)}")
            self.setVariable("LU1", AbsPos(start.x * 10, start.y * 10, start.z * 10, 0, 0, 0) + self.zeroPosition + self.upvector)
            self.setVariable("L1", AbsPos(start.x * 10, start.y * 10, start.z * 10, 0, 0, 0) + self.zeroPosition)
            self.setVariable("L2", AbsPos(end.x * 10, end.y * 10, end.z * 10, 0, 0, 0) + self.zeroPosition)
            self.setVariable("LU2", AbsPos(end.x * 10, end.y * 10, end.z * 10, 0, 0, 0) + self.zeroPosition + self.upvector)

            self.overrideSpeed(30)
            self.moveLinearTo("LU1", True, "P")
            time.sleep(1)

            self.overrideSpeed(10)
            self.moveLinearTo("L1", True, "P")
            time.sleep(0.4)

            self.moveLinearTo("L2", True, "P")
            time.sleep(self.calcLineLength(start, end) * 0.2)  # Adjust time based on length

            self.moveLinearTo("LU2", True, "P")
            time.sleep(0.4)

            
    def drawDXF(self, filename):
        doc = ezdxf.readfile(filename)
        msp = doc.modelspace()

        with self:
            self.servoOn()
            time.sleep(1)

        for entity in msp:
            if entity.dxftype() == 'LINE':
                start = entity.dxf.start
                end = entity.dxf.end
                print(f"Processing Line from {start} to {end} with length {self.calcLineLength(start, end)}")
                self.drawLine(start, end)

    def calcLineLength(self, start, end):
        # Calculate the length of a line segment
        return ((end.x - start.x) ** 2 + (end.y - start.y) ** 2 + (end.z - start.z) ** 2) ** 0.5




    

In [3]:
# Example usage
robot = Frace("/dev/ttyUSB0")
robot.connect()
robot.setupRobot()
robot.home()
print("Waiting two seconds before drawing...")
time.sleep(2)
print("Drawing DXF file...")
robot.drawDXF("CenteredHouse.dxf")

Connecting
QoK3F;3F;7,0;3,5,A,1E,32,46,64;MB4;PRM;RV-3SB;CRn-5xx;MELFA;05-07-14;Ver.K4a;ENG;COPYRIGHT(C)1999-2005 MITSUBISHI ELECTRIC CORPORATION ALL RIGHTS RESERVED;1;1;8;
QeR601000000
QoKRLNG;1;1
QeR601000000
QoK1.MB4;9104;13-03-2222:04:26;2;215808;12;;4;155;135256;0;154;876
QeR601000000
QoKX;419.93;Y;0.00;Z;322.02;A;-180.00;B;-0.08;C;-179.99;;7,0;100;0.00;00000000
QeR601000000
QoKMEXTL;0.00, 0.00, 0.00, 0.00, 0.00, 0.00;6
QeR601000000
QoK
Ended Connecting
Waiting two seconds before drawing...
Drawing DXF file...
Processing Line from (-0.9236603538738382, -3.987536745262332, 0.0) to (-0.9236603538738385, -1.615172697347589, 0.0) with length 2.3723640479147425
Drawing line from (-0.9236603538738382, -3.987536745262332, 0.0) to (-0.9236603538738385, -1.615172697347589, 0.0) with length 2.3723640479147425
Processing Line from (-2.294359312509168, -3.987536745262332, 0.0) to (-2.294359312509169, -1.615172697347589, 0.0) with length 2.3723640479147425
Drawing line from (-2.294359312509168

That much better. Now it is time for circles. Lets first just draw a circle with the robot

In [ ]:
import math
from gerrard import *


def circleCords(center, radius):
    """Calculates three evenly spaced points on a circle."""
    points = []
    for i in range(3):
        angle = 2 * math.pi * i / 3  # Divide the circle into three equal parts
        x = center.x + radius * math.cos(angle)
        y = center.y + radius * math.sin(angle)
        points.append((x, y))
    return points


def drawCircle(self:Frace, center, radius):
    """Draws a circle by calculating and moving to three points."""
    points = circleCords(center, radius)
    print(f"Circle points: {points}")

    with self:
        self.servoOn()
        time.sleep(1)


    for i, point in enumerate(points):
        print(f"Point {i+1}: {point}")
        pointer = self.zeroPosition + AbsPos(1, 1, 0, 0, 0, 0) + self.upvector
        print(f"Pointer {i+1}: {pointer}")
        self.drawLine(self.zeroPosition, pointer)
        time.sleep(10)

    with self:
        self.servoOn()
        time.sleep(1)

        start = AbsPos(points[0][0], points[0][1], 0, 0, 0, 0)
        end = AbsPos(points[1][0], points[1][1], 0, 0, 0, 0)

        self.setVariable("LU1", AbsPos(start.x * 10, start.y * 10, start.z * 10, 0, 0, 0) + self.zeroPosition + self.upvector)
        self.setVariable("L1", AbsPos(start.x * 10, start.y * 10, start.z * 10, 0, 0, 0) + self.zeroPosition)
        self.setVariable("L2", AbsPos(end.x * 10, end.y * 10, end.z * 10, 0, 0, 0) + self.zeroPosition)
        self.setVariable("LU2", AbsPos(end.x * 10, end.y * 10, end.z * 10, 0, 0, 0) + self.zeroPosition + self.upvector)

        self.setVariable("HU1", AbsPos(round(points[0][0]*10), round(points[0][1]*10), 0, 0, 0, 0) + self.zeroPosition + self.upvector)
        time.sleep(0.2)
        self.setVariable("H1", AbsPos(round(points[0][0]*10), round(points[0][1]*10), 0, 0, 0, 0) + self.zeroPosition)
        time.sleep(0.2)
        self.setVariable("H2", AbsPos(round(points[1][0]*10), round(points[1][1]*10), 0, 0, 0, 0) + self.zeroPosition)
        time.sleep(0.2)
        self.setVariable("H3", AbsPos(round(points[2][0]*10), round(points[2][1]*10), 0, 0, 0, 0) + self.zeroPosition)
        time.sleep(0.2)
        self.setVariable("HU3", AbsPos(round(points[2][0]*10), round(points[2][1]*10), 0, 0, 0, 0) + self.zeroPosition + self.upvector)
        time.sleep(0.2)

        time.sleep(5)
        self.overrideSpeed(30)
        self.moveLinearTo("LU1", True, "P")
        print("Moving to L1")
        time.sleep(1)

        self.overrideSpeed(10)
        self.moveLinearTo("H1", True, "P")
        print("Moving to H1")
        time.sleep(2)

        self.moveLinearTo("H2", True, "P")
        print("Moving to H2")
        time.sleep(2)

        self.moveLinearTo("H3", True, "P")
        print("Moving to H3")
        time.sleep(2)

        # self.MoveCircle("PH1", "PH2", "PH3")
        self.executeCommand(f"MVC PH1, PH2, PH3", True)
        print("Moving in circle")
        time.sleep(3)

        self.moveLinearTo("HU3", True, "P")
        print("Moving to HU3")
        time.sleep(0.4)

# Example usage
robot = Frace("/dev/ttyUSB0")
robot.connect()
robot.setupRobot()
robot.home()
print("Waiting two seconds before drawing...")
time.sleep(2)
print("Drawing circle...")
drawCircle(robot, AbsPos(0, 0, 0, 0, 0, 0), 1)  # Center at (0, 0) with radius 50

In [6]:
import ezdxf

with newRobot:

    print(f"Processing Line from {entity.dxf.start} to {entity.dxf.end}")
    start = {"x": 1, "y": 1, "z": 1}  # Replace with actual start coordinates
    mid = {"x": 2, "y": 2, "z": 2}  # Replace with actual mid coordinates
    end = {"x": 3, "y": 1, "z": 2}  # Replace with actual end coordinates
    
    newRobot.setVariable("L1", AbsPos(start.x*10, start.y*10, start.z*10, 0, 0, 0) + zeroPosition )
    time.sleep(0.2)
    newRobot.setVariable("LU1", AbsPos(start.x*10, start.y*10, start.z*10, 0, 0, 0) + zeroPosition + upvector)
    time.sleep(0.2)
    newRobot.setVariable("L2", AbsPos(mid.x*10, mid.y*10, mid.z*10, 0, 0, 0) + zeroPosition)
    time.sleep(0.2)
    newRobot.setVariable("LU2", AbsPos(mid.x*10, mid.y*10, mid.z*10, 0, 0, 0) + zeroPosition + upvector)
    time.sleep(0.2)
    newRobot.setVariable("L3", AbsPos(end.x*10, end.y*10, end.z*10, 0, 0, 0) + zeroPosition)
    time.sleep(0.2)
    newRobot.setVariable("LU3", AbsPos(end.x*10, end.y*10, end.z*10, 0, 0, 0) + zeroPosition + upvector)
    time.sleep(0.2)

    newRobot.overrideSpeed(20)
    newRobot.moveLinearTo("LU1", True, "P")
    time.sleep(0.5)

    newRobot.overrideSpeed(10)
    time.sleep(0.5)
    newRobot.moveLinearTo("L1", True, "P")
    time.sleep(1)
    
    newRobot.moveLinearTo("L2", True, "P")
    time.sleep(3)
    newRobot.moveLinearTo("LU2", True, "P")
    time.sleep(0.5)


Processing Line from (-9.200575947761536, 3.44587154686451, 0.0) to (-7.988034188747406, 3.44587154686451, 0.0)


AttributeError: 'dict' object has no attribute 'x'